# Qwen on P5 (text purpose): dev + train runs, system check, machine comparison
Runs `qwen3:8b` through a local Ollama server as a standalone span detector on the shared **dev** (machine comparison, tuning) and
**train** (pre-tuning benchmark, error mining) splits, scores it against the gold, and records a hardware/latency check.
**P5 only**: keep public figures, organizations, places, non-identifying dates; redact private persons, contacts, IDs, street
addresses, personal URLs, usernames, dates of birth, demographics.

**Test is locked** until the pipeline is frozen. Run the sections in order: Setup, Helpers, Data check, System check, DEV run,
Dev metrics, Compare machines on dev (optional), TRAIN run, Train benchmark, Export.
Data (`data/processed/`) and results (`llm_p5/results/`) are gitignored. Results files contain document text: send them privately, never via git.

## Setup

In [ ]:
from pathlib import Path

HERE = Path.cwd()
ROOT = HERE.parent if HERE.name == "llm_p5" else HERE          # repo root (works from llm_p5/ or the root)
LLM_DIR = ROOT / "llm_p5"
RESULTS_DIR, REFERENCE_DIR = LLM_DIR / "results", LLM_DIR / "reference"
INCOMING_DIR = RESULTS_DIR / "incoming"                          # results files received from other machines
CASES_PATH = LLM_DIR / "system_check_cases.jsonl"
DATA_DIR = ROOT / "data" / "processed"            # dev.jsonl, train.jsonl, test.jsonl (unzip the shared data here)

MODEL_TAG = "qwen3:8b"
OLLAMA_URL = "http://localhost:11434"
WINDOW, OVERLAP = 1200, 100          # characters; windows end at sentence boundaries
NUM_CTX, NUM_PREDICT = 4096, 900
DEADLINE = 120                       # seconds, wall clock per request
SEED = 42

MACHINE_LABEL = None                 # e.g. "win_rtx4070": used in file names instead of the detected chip_ramgb
HARDWARE_OVERRIDE = {"cpu": None, "ram_gb": None, "gpu": None, "vram_gb": None, "os": None}   # non-None values win over detection

# Prompt freeze: the hash covers the system prompt + output schema. A mismatch stops the run.
PROMPT_VERSION = "v0.1"
EXPECTED_PROMPT_HASH = "06f049e45980"
ALLOW_PROMPT_CHANGE = False          # True = run anyway; every row is stamped prompt_modified=true

RUN_SPLITS = ["dev", "train"]        # dev always runs before train
ALLOW_TEST = False                   # test is locked until the pipeline is frozen
SAMPLE_SIZES = {"dev": {"crapii": 150, "tab": 20}, "train": {"crapii": 300, "tab": 25},
                "test": {"crapii": 150, "tab": 20}}      # test is only used if ALLOW_TEST=True
SAMPLE_IDS_FILE = None               # json list of doc_ids: overrides sampling (e.g. same docs as Presidio/GLiNER)
MAX_HOURS_DEV, MAX_HOURS_TRAIN = None, None   # wall-clock budget per split: stop starting new docs after this; re-run resumes
TAB_SCOPE = "direct+quasi"           # "direct+quasi" | "direct": headline TAB gold scope
TAB_ANNOTATOR = "first"              # "first" | "union"
N_BOOT = 1000
OTHER_DEV_FILE = None                # dev results file from another machine (filename in llm_p5/results/incoming/)
OTHER_CHECK_FILE = None              # that machine's system_check json (optional: adds its GPU share)

> **Samples.** Per source and split the order is fixed by the seed (CRAPII: docs with gold spans and docs without, interleaved 1:1; TAB: one
> permutation); a sample of size *n* is the first *n* of that order, so a larger size extends a smaller one and an early stop still leaves a
> stratified random subset. Resume stays valid. Only change a sample size by increasing it.

## Helpers
Pure functions (cells tagged `helpers` are exec'd by `llm_p5/tests`). Nothing here talks to Ollama until called.

### Windows, string matching, merge

In [ ]:
import hashlib, http.client, json, locale, os, platform, random, re, socket, subprocess, sys, threading, time, urllib.parse, urllib.request
import numpy as np
import pandas as pd

UNIFIED_LABELS = ["PERSON_PRIVATE", "PERSON_PUBLIC", "EMAIL", "PHONE", "ID_NUM", "STREET_ADDRESS", "DOB",
                  "DATE_OTHER", "LOCATION_OTHER", "URL_PERSONAL", "OTHER_PII", "DEMOGRAPHIC", "ORG"]
P5_KEEP = {"PERSON_PUBLIC", "ORG", "DATE_OTHER", "LOCATION_OTHER"}   # everything else is REDACT under P5_text

_SENT_END = re.compile(r"[.!?][\"')\]]*\s|\n")


def make_windows(text, size=1200, overlap=100):
    """Sentence-boundary windows [(start, end)] in document offsets; consecutive windows overlap by ~`overlap` chars."""
    n = len(text)
    if n == 0:
        return []
    out, start = [], 0
    while True:
        end = min(start + size, n)
        if end < n:
            lo = start + size // 2
            cut = None
            for m in _SENT_END.finditer(text, lo, end):
                cut = m.end()
            if cut is None:
                ws = max(text.rfind(c, lo, end) for c in (" ", "\n", "\t"))
                cut = ws + 1 if ws >= 0 else end
            end = cut
        out.append((start, end))
        if end >= n:
            break
        nxt = max(end - overlap, start + 1)
        while nxt < end and not text[nxt - 1].isspace():   # begin the next window at a word start
            nxt += 1
        start = nxt
    return out


_QUOTE_MAP = {"\u2018": "'", "\u2019": "'", "\u201a": "'", "\u201b": "'", "\u2032": "'", "\u02bc": "'",
              "\u201c": '"', "\u201d": '"', "\u201e": '"', "\u201f": '"', "\u2033": '"'}


def norm_map(text):
    """Offset-preserving normalisation: whitespace runs (incl. newlines) -> one space, curly quotes -> straight.
    Returns (normalised, starts, ends): normalised[i] covers text[starts[i]:ends[i]]."""
    out, st, en, i, n = [], [], [], 0, len(text)
    while i < n:
        c = text[i]
        if c.isspace():
            j = i + 1
            while j < n and text[j].isspace():
                j += 1
            out.append(" ")
            st.append(i)
            en.append(j)
            i = j
        else:
            out.append(_QUOTE_MAP.get(c, c))
            st.append(i)
            en.append(i + 1)
            i += 1
    return "".join(out), st, en


def norm_key(t):
    return norm_map(t or "")[0].strip()


def match_strings_ex(wtext, items, offset=0):
    """Map returned strings to ALL word-bounded occurrences in the window, longest first, no overlaps; tolerant to
    whitespace runs/newlines and curly vs straight quotes. items: [{text,label,decision}].
    Returns (preds in document offsets, [strings with no match])."""
    ntext, st, en = norm_map(wtext)
    best, raw = {}, {}
    for it in items:
        t = it.get("text") or ""
        k = norm_key(t)
        if not k:
            continue
        cur = best.get(k)
        if cur is None or (it.get("decision") == "REDACT" and cur.get("decision") != "REDACT"):
            best[k] = {"label": it.get("label"), "decision": it.get("decision")}
        raw.setdefault(k, t.strip())
    taken, preds, unmatched = [], [], []
    for k in sorted(best, key=lambda x: (-len(x), x)):
        pat = re.compile(r"(?<!\w)" + re.escape(k) + r"(?!\w)")
        seen = False
        for m in pat.finditer(ntext):
            seen = True
            s, e = st[m.start()], en[m.end() - 1]
            if any(s < e2 and s2 < e for s2, e2 in taken):
                continue
            taken.append((s, e))
            preds.append({"start": s + offset, "end": e + offset, "text": wtext[s:e], **best[k]})
        if not seen:
            unmatched.append(raw[k])
    return sorted(preds, key=lambda p: p["start"]), unmatched


def match_strings(wtext, items, offset=0):
    """match_strings_ex with the number of unmatched strings instead of the list."""
    preds, unmatched = match_strings_ex(wtext, items, offset)
    return preds, len(unmatched)


def merge_preds(preds):
    """Union across windows: identical (start, end) collapse, REDACT wins over KEEP."""
    d = {}
    for p in preds:
        k = (p["start"], p["end"])
        c = d.get(k)
        if c is None or (p["decision"] == "REDACT" and c["decision"] != "REDACT"):
            d[k] = p
    return sorted(d.values(), key=lambda p: (p["start"], p["end"]))


def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def utf8_guard(log=print, utf8_mode=None, encoding=None):
    """Warn loudly when Python is not in UTF-8 mode (Windows default is a legacy code page). Returns the message or None.
    All file I/O in this notebook passes encoding="utf-8" explicitly, so results do not depend on it."""
    mode = sys.flags.utf8_mode if utf8_mode is None else utf8_mode
    if mode:
        return None
    enc = encoding or locale.getpreferredencoding(False)
    msg = (f"sys.flags.utf8_mode == 0 (locale.getpreferredencoding() = {enc!r}). Set PYTHONUTF8=1 before starting Jupyter. "
           "All notebook file I/O is explicitly UTF-8 anyway, so results are not affected here; "
           "code you add that calls open() without encoding= would be.")
    log("!" * 78)
    log("WARNING: " + msg)
    log("!" * 78)
    return msg


UTF8_WARNING = utf8_guard()

### Guardrailed Ollama client, hardware info

In [ ]:
REQUEST_TIMEOUT = 120


LOOP_RUN, LOOP_TOTAL = 3, 5      # repetition guard: abort after 3 repeated objects in a row or 5 in total


class _ObjScan:
    """Incremental scan of the streamed reply {"spans":[{...},...]}: feed() returns the {...} items completed by a chunk."""

    def __init__(self):
        self.depth, self.in_str, self.esc, self.buf = 0, False, False, []

    def feed(self, chunk):
        done = []
        for ch in chunk:
            if self.in_str:
                if self.depth >= 2:
                    self.buf.append(ch)
                if self.esc:
                    self.esc = False
                elif ch == "\\":
                    self.esc = True
                elif ch == '"':
                    self.in_str = False
                continue
            if ch == "{":
                self.depth += 1
                if self.depth == 2:
                    self.buf = []
            if self.depth >= 2:
                self.buf.append(ch)
            if ch == '"':
                self.in_str = True
            elif ch == "}":
                if self.depth == 2:
                    try:
                        obj = json.loads("".join(self.buf))
                        if isinstance(obj, dict):
                            done.append(obj)
                    except ValueError:
                        pass
                self.depth = max(self.depth - 1, 0)
        return done


def _stream(body, url, timeout, t0, num_predict, out):
    """Stream one request into `out`; raises OSError subclasses on connection failures."""
    u = urllib.parse.urlsplit(url)
    conn = http.client.HTTPConnection(u.hostname, u.port, timeout=timeout)
    state = {"timed_out": False}
    sock = []  # http.client drops conn.sock for close-delimited responses, so keep our own reference

    def kill():  # watchdog: abort the connection at the deadline, even if the server sends nothing
        state["timed_out"] = True
        try:
            sock[0].shutdown(socket.SHUT_RDWR)
        except (OSError, IndexError):
            pass
        conn.close()

    parts, chunks = [], 0
    scan, seen, run = _ObjScan(), set(), 0
    try:
        conn.connect()
        sock.append(conn.sock)
        timer = threading.Timer(max(timeout - (time.perf_counter() - t0), 0.01), kill)
        timer.daemon = True
        timer.start()
        try:
            conn.request("POST", u.path or "/", json.dumps(body).encode(), {"Content-Type": "application/json"})
            resp = conn.getresponse()
            if resp.status != 200:
                out["error"] = f"HTTP {resp.status}: {resp.reason}"
                return
            while True:
                line = resp.readline()
                if not line:
                    if not state["timed_out"] and out["done_reason"] is None:
                        out["error"] = "incomplete stream"
                    break
                obj = json.loads(line)
                if obj.get("error"):
                    out["error"] = f"server: {obj['error']}"
                    break
                parts.append((obj.get("message") or {}).get("content") or "")
                chunks += 1
                for o in scan.feed(parts[-1]):
                    if not isinstance(o.get("text"), str) or "label" not in o:
                        continue          # not a {text,label,decision} item (e.g. review-mode ids): no guard
                    key = (o.get("text"), o.get("label"), o.get("decision"))
                    if key in seen:       # the prompt asks for each distinct string once: a repeat
                        out["repeats"] += 1
                        run += 1
                        if run >= LOOP_RUN or out["repeats"] >= LOOP_TOTAL:
                            out["loop"] = True
                    else:
                        seen.add(key)
                        out["objects"].append(o)
                        run = 0
                if out["loop"]:           # abort: same socket shutdown as the deadline watchdog
                    try:
                        sock[0].shutdown(socket.SHUT_RDWR)
                    except (OSError, IndexError):
                        pass
                    break
                if obj.get("done"):
                    out["done_reason"] = obj.get("done_reason") or "stop"
                    out["eval_count"], out["eval_duration_ns"] = obj.get("eval_count"), obj.get("eval_duration")
                    out["load_duration_ns"] = obj.get("load_duration")
                    out["prompt_eval_duration_ns"], out["total_duration_ns"] = obj.get("prompt_eval_duration"), obj.get("total_duration")
                    out["prompt_eval_count"] = obj.get("prompt_eval_count")
                    break
                if chunks > num_predict:  # a chunk is >= 1 token: the server ignored the cap
                    out["done_reason"] = "length"
                    break
        except (OSError, ValueError, http.client.HTTPException) as e:
            state["timed_out"] |= isinstance(e, TimeoutError)  # socket timeout racing the watchdog
            if not state["timed_out"]:
                out["error"] = f"{type(e).__name__}: {e}"
        finally:
            timer.cancel()
    finally:
        conn.close()
    out["raw"] = "".join(parts) or None
    if out["eval_count"] is None:
        out["eval_count"] = chunks
    if state["timed_out"]:
        out["error"] = "timeout"
    elif out["loop"]:
        out["error"] = "loop"
    elif out["done_reason"] == "length":
        out["truncated"], out["error"] = True, "truncated"
    if out["loop"] or (out["truncated"] and out["repeats"]):
        out["cause"] = "loop"
    elif out["truncated"]:
        out["cause"] = "dense"            # hit the cap with only unique objects


def chat(model, messages, schema, url, num_ctx=4096, seed=42, timeout=REQUEST_TIMEOUT, num_predict=900):
    """POST to Ollama /api/chat (url = full endpoint). Streams internally, caps tokens, enforces a wall-clock deadline,
    think=false, temperature 0. No retry on timeout, truncation or bad JSON; one retry only on a refused connection.
    Returns {raw, parsed, valid_json, latency_s, eval_count, eval_duration_ns, load_duration_ns, prompt_eval_duration_ns,
    total_duration_ns, prompt_eval_count, tokens_per_s, error, done_reason, truncated,
    objects (unique complete items streamed so far), repeats, loop (aborted by the repetition guard), cause (none|loop|dense)}."""
    body = {
        "model": model, "messages": messages, "format": schema, "stream": True, "think": False,
        "keep_alive": "5m", "options": {"temperature": 0, "num_ctx": num_ctx, "seed": seed, "num_predict": num_predict},
    }
    out = {"raw": None, "parsed": None, "valid_json": False, "latency_s": None, "eval_count": None,
           "eval_duration_ns": None, "load_duration_ns": None, "prompt_eval_duration_ns": None,
           "total_duration_ns": None, "prompt_eval_count": None, "tokens_per_s": None, "error": None,
           "done_reason": None, "truncated": False, "objects": [], "repeats": 0, "loop": False, "cause": "none"}
    t0 = time.perf_counter()
    for attempt in (0, 1):
        try:
            _stream(body, url, timeout, t0, num_predict, out)
            break
        except ConnectionRefusedError as e:
            out["error"] = f"{type(e).__name__}: {e}"
        except (OSError, http.client.HTTPException) as e:  # connect timeout, DNS, reset
            out["error"] = "timeout" if isinstance(e, TimeoutError) else f"{type(e).__name__}: {e}"
            break
    out["latency_s"] = time.perf_counter() - t0
    if out["eval_count"]:
        dur = (out["eval_duration_ns"] or 0) / 1e9 or out["latency_s"]
        out["tokens_per_s"] = out["eval_count"] / dur if dur else None
    if out["error"] is None and out["raw"] is not None:
        try:
            parsed = json.loads(out["raw"])
            if isinstance(parsed, dict):
                out["parsed"], out["valid_json"] = parsed, True
        except ValueError:
            pass
    return out


def make_chat_fn(model, base_url, num_ctx=4096, num_predict=900, deadline=120, seed=42):
    """Bind the settings; returns chat_fn(messages, schema) -> chat() result."""
    url = base_url.rstrip("/") + "/api/chat"
    return lambda messages, schema: chat(model, messages, schema, url, num_ctx=num_ctx, seed=seed,
                                         timeout=deadline, num_predict=num_predict)


def _http_json(url, data=None, timeout=10):
    req = urllib.request.Request(url, data=None if data is None else json.dumps(data).encode(),
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read())


def ollama_version(base_url):
    try:
        return _http_json(base_url.rstrip("/") + "/api/version").get("version")
    except Exception:
        return None


def model_info(tag, base_url):
    """{digest, size} for a pulled model tag (None values if the server is unreachable or the tag is missing)."""
    try:
        for m in _http_json(base_url.rstrip("/") + "/api/tags").get("models", []):
            if m.get("name") == tag or m.get("model") == tag or m.get("name") == tag + ":latest":
                return {"digest": m.get("digest"), "size": m.get("size")}
    except Exception:
        pass
    return {"digest": None, "size": None}


def gpu_share(tag, base_url):
    """size_vram / size of the loaded model from /api/ps (None if not loaded)."""
    try:
        for m in _http_json(base_url.rstrip("/") + "/api/ps").get("models", []):
            if m.get("name") == tag or m.get("model") == tag:
                return (m["size_vram"] / m["size"]) if m.get("size") else None
    except Exception:
        pass
    return None


def _sh(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=10).stdout.strip()
    except Exception:
        return ""


def _read_text(path):
    with open(path, encoding="utf-8", errors="replace") as f:
        return f.read()


def _ps(sh, expr):
    return sh(["powershell", "-NoProfile", "-NonInteractive", "-Command", expr])


def parse_nvidia_smi(out):
    """First line of `nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader` -> {name, vram_gb, driver}, or None."""
    line = next((ln for ln in (out or "").splitlines() if ln.strip()), "")
    parts = [p.strip() for p in line.split(",")]
    if len(parts) < 3 or not parts[0]:
        return None
    m = re.search(r"[\d.]+", parts[1])
    return {"name": parts[0], "vram_gb": round(float(m.group()) / 1024) if m else None, "driver": parts[2] or None}


def hw_info(override=None, sh=_sh, system=None, reader=_read_text):
    """OS, chip, RAM, GPU, Python. Deliberately no hostname, username or paths.
    override: {cpu, ram_gb, gpu, vram_gb, os}; non-None values win over detection."""
    system = system or platform.system()
    chip, ram = "", None
    if system == "Darwin":
        chip = sh(["sysctl", "-n", "machdep.cpu.brand_string"])
        mem = sh(["sysctl", "-n", "hw.memsize"])
        ram = round(int(mem) / 2**30) if mem.isdigit() else None
    elif system == "Windows":
        chip = _ps(sh, "(Get-CimInstance Win32_Processor | Select-Object -First 1).Name").strip()
        mem = _ps(sh, "(Get-CimInstance Win32_ComputerSystem).TotalPhysicalMemory").strip()
        ram = round(int(mem) / 2**30) if mem.isdigit() else None
    elif system == "Linux":
        try:
            for line in reader("/proc/cpuinfo").splitlines():
                if line.lower().startswith("model name"):
                    chip = line.split(":", 1)[1].strip()
                    break
            for line in reader("/proc/meminfo").splitlines():
                if line.startswith("MemTotal"):
                    ram = round(int(line.split()[1]) / 2**20)
                    break
        except (OSError, ValueError, IndexError):
            pass
    gpu = parse_nvidia_smi(sh(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"]))
    gpu = gpu or {"name": None, "vram_gb": None, "driver": None}
    hw = {"os": platform.platform(), "chip": chip or platform.processor() or platform.machine() or "unknown",
          "ram_gb": ram, "gpu": gpu, "python": platform.python_version()}
    ov = override or {}
    for key, dst in (("cpu", "chip"), ("ram_gb", "ram_gb"), ("os", "os")):
        if ov.get(key) is not None:
            hw[dst] = ov[key]
    if ov.get("gpu") is not None:
        hw["gpu"] = {**gpu, "name": ov["gpu"]}
    if ov.get("vram_gb") is not None:
        hw["gpu"] = {**hw["gpu"], "vram_gb": ov["vram_gb"]}
    return hw


def slug(s):
    return re.sub(r"[^a-z0-9]+", "_", str(s).lower()).strip("_") or "unknown"


def machine_tag(hw, label=None):
    """File-name tag: MACHINE_LABEL if set, else chip_ramgb. Only [a-z0-9_]."""
    if label:
        return slug(label)
    return f"{slug(hw.get('chip'))}_{hw['ram_gb']}gb" if hw.get("ram_gb") else slug(hw.get("chip"))


def git_info():
    try:
        commit = _sh(["git", "rev-parse", "--short", "HEAD"]) or None
        dirty = bool(_sh(["git", "status", "--porcelain", "--untracked-files=no"]))
    except Exception:
        commit, dirty = None, None
    return {"git_commit": commit, "git_dirty": dirty}

### Prompts (P5 policy inlined) and prompt hash

In [ ]:
LABEL_DEFS = {
    "DOB": "a person's date of birth",
    "DATE_OTHER": "any other date",
    "PERSON_PUBLIC": "a well-known public or historical figure",
    "PERSON_PRIVATE": "any other person",
    "STREET_ADDRESS": "a home-level address",
    "LOCATION_OTHER": "city, region, venue or non-home place",
    "DEMOGRAPHIC": "age, gender or similar",
    "ORG": "an organization",
    "ID_NUM": "an identification number",
    "EMAIL": "an email address",
    "PHONE": "a phone number",
    "URL_PERSONAL": "a personal URL",
    "OTHER_PII": "any other personally identifying detail",
}
_DEFS = "\n".join(f"- {k}: {v}" for k, v in LABEL_DEFS.items())
_POLICY = """Policy (purpose P5, text that will be shared as ordinary text):
- KEEP: public or historical figures, organizations, cities, regions and venues, dates that do not identify a person.
- REDACT: private persons, email addresses, phone numbers, identification numbers, street or home addresses, personal URLs, usernames and handles, dates of birth, and demographic details (age, gender or similar) of an individual."""

SYSTEM_STANDALONE = f"""You are a privacy-redaction detector. Read the text and list every distinct person, place, organization, date, contact detail and identifier string in it, once each, exactly as written. For each string give one label and one decision, REDACT or KEEP, following the policy.

{_POLICY}

Labels:
{_DEFS}

Rules:
- Copy each string exactly as it appears in the text; do not paraphrase or add words.
- Copy each string exactly as it appears in the text, including capitalization and punctuation.
- List each distinct string once, even if it appears several times.
- Do not give offsets or positions.
- If nothing qualifies, return an empty list.

Answer with JSON: {{"spans": [{{"text", "label", "decision"}}]}}."""

SYSTEM_REVIEW = f"""You are a privacy-redaction reviewer. You are given a text and a numbered list of candidate spans found by a detector. For each candidate give the decision REDACT or KEEP following the policy. You may also list personal-information strings in the text that are missing from the candidate list (copied exactly as written, with a label and a decision).

{_POLICY}

Labels:
{_DEFS}

Answer with JSON: {{"decisions": [{{"id", "decision"}}], "missed": [{{"text", "label", "decision"}}]}}."""

_DECISION = {"type": "string", "enum": ["REDACT", "KEEP"]}
_LABEL = {"type": "string", "enum": UNIFIED_LABELS}
SCHEMA_STANDALONE = {"type": "object", "required": ["spans"], "properties": {"spans": {"type": "array", "items": {
    "type": "object", "required": ["text", "label", "decision"],
    "properties": {"text": {"type": "string"}, "label": _LABEL, "decision": _DECISION}}}}}
SCHEMA_REVIEW = {"type": "object", "required": ["decisions", "missed"], "properties": {
    "decisions": {"type": "array", "items": {"type": "object", "required": ["id", "decision"],
                                             "properties": {"id": {"type": "integer"}, "decision": _DECISION}}},
    "missed": {"type": "array", "items": {"type": "object", "required": ["text", "label", "decision"],
                                          "properties": {"text": {"type": "string"}, "label": _LABEL,
                                                         "decision": _DECISION}}}}}


def prompt_hash(mode="standalone"):
    system, schema = (SYSTEM_REVIEW, SCHEMA_REVIEW) if mode == "review" else (SYSTEM_STANDALONE, SCHEMA_STANDALONE)
    return hashlib.sha1((system + json.dumps(schema, sort_keys=True)).encode()).hexdigest()[:12]


def build_messages(window_text):
    return [{"role": "system", "content": SYSTEM_STANDALONE}, {"role": "user", "content": f"TEXT:\n{window_text}"}]


def build_review_messages(window_text, cands):
    """cands: [{text,label,score}] in display order (ids are 1..k). Only text/label/score are shown: no gold info."""
    rows = ["id | text | detector_label | score"]
    for i, c in enumerate(cands, 1):
        score = c.get("score")
        rows.append(f"{i} | {c.get('text')} | {c.get('label')} | {'' if score is None else round(float(score), 3)}")
    user = f"TEXT:\n{window_text}\n\nCANDIDATES:\n" + "\n".join(rows)
    return [{"role": "system", "content": SYSTEM_REVIEW}, {"role": "user", "content": user}]

### Detection (resume-safe, time budget)

In [ ]:
def _items(parsed, key="spans"):
    """Valid span items from a parsed reply, or None when the reply has the wrong shape."""
    lst = parsed.get(key) if isinstance(parsed, dict) else None
    if not isinstance(lst, list):
        return None
    return [x for x in lst if isinstance(x, dict) and isinstance(x.get("text"), str)
            and x.get("decision") in ("REDACT", "KEEP")]


def _ns_to_s(ns):
    return None if ns is None else ns / 1e9


def _sum_opt(xs):
    xs = [x for x in xs if x is not None]
    return sum(xs) if xs else None


TIMING_KEYS = ("prompt_eval_s", "load_s", "gen_s", "total_s", "prompt_eval_count")


PIPELINE_VERSION = "p1"     # part of the resume key: rows from other pipeline versions are never reused


def _dedupe_items(items):
    seen, out = set(), []
    for x in items:
        k = (x.get("text"), x.get("label"), x.get("decision"))
        if k not in seen:
            seen.add(k)
            out.append(x)
    return out


def split_point(wt):
    """Sentence boundary (offset into wt) nearest the middle; whitespace, then the middle itself, as fallbacks. None if too short."""
    n = len(wt)
    if n < 2:
        return None
    cands = [m.end() for m in _SENT_END.finditer(wt) if 0 < m.end() < n]
    if not cands:
        cands = [m.end() for m in re.finditer(r"\s+", wt) if 0 < m.end() < n]
    return min(cands, key=lambda c: (abs(c - n / 2), c)) if cands else n // 2


def _call_standalone(wt, off, chat_fn):
    """One model call on wt (document offset `off`). Truncated or loop-aborted replies are salvaged: every complete
    object streamed so far, deduped. Returns (result, window-stats dict, items or None, preds, unmatched strings, bad)."""
    r = chat_fn(build_messages(wt), SCHEMA_STANDALONE)
    trunc, loop = bool(r.get("truncated")), bool(r.get("loop"))
    bad = trunc or loop
    cause = "none" if not bad else ("loop" if loop else (r.get("cause") or "dense"))
    if bad:
        items = _dedupe_items(_items({"spans": r.get("objects") or []}) or [])
    else:
        items = _items(r.get("parsed")) if r.get("valid_json") else None
    preds, unm = match_strings_ex(wt, items or [], off)
    w = {"latency_s": r.get("latency_s"), "tokens_per_s": r.get("tokens_per_s"), "eval_count": r.get("eval_count"),
         "valid": items is not None and not bad, "truncated": trunc, "cause": cause,
         "n_spans_salvaged": len(items) if bad else 0,
         "load_s": _ns_to_s(r.get("load_duration_ns")), "prompt_eval_s": _ns_to_s(r.get("prompt_eval_duration_ns")),
         "gen_s": _ns_to_s(r.get("eval_duration_ns")), "total_s": _ns_to_s(r.get("total_duration_ns")),
         "prompt_eval_count": r.get("prompt_eval_count")}
    if bad:
        w["raw_partial"] = (r.get("raw") or "")[:2000]     # contains document text: stays in llm_p5/results/
    return r, w, items, preds, unm, bad


def _standalone_window(wt, s, chat_fn):
    """One window with recovery: if the call is truncated or loop-aborted, salvage it, then run both halves once
    (split at the sentence boundary nearest the middle; depth 1, no further splitting) and union everything.
    Returns (window-stat entries [main, half1, half2], preds, error strings)."""
    r, w, items, preds, unm, bad = _call_standalone(wt, s, chat_fn)
    entries, allp, calls, errs = [w], list(preds), [(items, unm)], [r.get("error")]
    w["split"], w["failed_after_split"] = False, False
    if bad:
        cut = split_point(wt)
        if cut is None:
            w["failed_after_split"] = True
        else:
            w["split"] = True
            for half, (a, b) in enumerate(((0, cut), (cut, len(wt))), 1):
                rh, wh, ih, ph, uh, bh = _call_standalone(wt[a:b], s + a, chat_fn)
                wh.update({"half": half, "split": False, "failed_after_split": False, "unmatched": []})
                entries.append(wh)
                allp += ph
                calls.append((ih, uh))
                errs.append(rh.get("error"))
                w["failed_after_split"] |= bh      # half also truncated/looped: salvaged and counted, never split again
    ret, unc, raw = {}, {}, {}
    for its, un in calls:
        for k in {norm_key(x["text"]) for x in its or []} - {""}:
            ret[k] = ret.get(k, 0) + 1
        for t in un:
            k = norm_key(t)
            unc[k], raw[k] = unc.get(k, 0) + 1, t
    # a string counts as unmatched only if no call (window or half) found it
    w["unmatched"] = [raw[k][:80] for k in unc if unc[k] >= ret.get(k, 0)]
    w["n_returned"] = len(ret)
    return entries, allp, errs


def detect_doc(doc, chat_fn, window=1200, overlap=100, mode="standalone", candidates=None):
    """Run the LLM over every window of one doc; returns a result row (no metadata)."""
    text = doc["text"]
    wins = make_windows(text, window, overlap)
    cands = sorted(candidates or [], key=lambda c: (c["start"], c["end"]))
    reviewed = set()
    preds, wstats, errors = [], [], {}
    n_trunc = n_inv = n_unm = 0
    t0 = time.perf_counter()
    for s, e in wins:
        wt = text[s:e]
        if mode != "review":
            entries, p, errs = _standalone_window(wt, s, chat_fn)
            w0 = entries[0]
            n_trunc += w0["truncated"]
            n_inv += (not w0["valid"]) and w0["cause"] == "none"
            n_unm += len(w0["unmatched"])
            for er in errs:
                if er:
                    errors[er] = errors.get(er, 0) + 1
            wstats += entries
            preds += p
            continue
        idx = [i for i, c in enumerate(cands) if c["start"] >= s and c["end"] <= e]
        msgs, schema = build_review_messages(wt, [cands[i] for i in idx]), SCHEMA_REVIEW
        r = chat_fn(msgs, schema)
        ok = bool(r.get("valid_json")) and not r.get("truncated")
        items = _items(r.get("parsed"), "missed") if ok else None
        if r.get("truncated"):
            n_trunc += 1
        elif items is None:
            n_inv += 1
        if r.get("error"):
            errors[r["error"]] = errors.get(r["error"], 0) + 1
        wstats.append({"latency_s": r.get("latency_s"), "tokens_per_s": r.get("tokens_per_s"),
                       "eval_count": r.get("eval_count"), "valid": items is not None,
                       "truncated": bool(r.get("truncated")),
                       "load_s": _ns_to_s(r.get("load_duration_ns")), "prompt_eval_s": _ns_to_s(r.get("prompt_eval_duration_ns")),
                       "gen_s": _ns_to_s(r.get("eval_duration_ns")), "total_s": _ns_to_s(r.get("total_duration_ns")),
                       "prompt_eval_count": r.get("prompt_eval_count")})
        if items is None:
            continue
        dec = {d.get("id"): d.get("decision") for d in _items_dec(r["parsed"])}
        for k, i in enumerate(idx, 1):
            c = cands[i]
            reviewed.add(i)
            preds.append({"start": c["start"], "end": c["end"], "text": c.get("text", text[c["start"]:c["end"]]),
                          "label": c.get("label"), "decision": dec.get(k, "REDACT")})
        p, u = match_strings(wt, items, s)
        preds += p
        n_unm += u
    if mode == "review":   # candidates no window fully contained: keep the detector's call (REDACT)
        for i, c in enumerate(cands):
            if i not in reviewed:
                preds.append({"start": c["start"], "end": c["end"], "text": c.get("text", ""),
                              "label": c.get("label"), "decision": "REDACT"})
    main = [w for w in wstats if not w.get("half")]
    return {"doc_id": doc["doc_id"], "source": doc.get("source"), "split": doc.get("split"), "mode": mode,
            "n_windows": len(wins), "n_truncated": n_trunc, "n_invalid": n_inv, "n_unmatched": n_unm,
            "n_loop": sum(w.get("cause") == "loop" for w in main), "n_dense": sum(w.get("cause") == "dense" for w in main),
            "n_split": sum(bool(w.get("split")) for w in main),
            "n_failed_after_split": sum(bool(w.get("failed_after_split")) for w in main),
            "n_salvaged": sum(w.get("n_spans_salvaged", 0) for w in wstats),
            "preds": merge_preds(preds), "latency_s": time.perf_counter() - t0, "windows": wstats, "errors": errors,
            **{k: _sum_opt(w.get(k) for w in wstats) for k in TIMING_KEYS}}   # per-row sums over windows and halves (None = server sent none)


def _items_dec(parsed):
    lst = parsed.get("decisions") if isinstance(parsed, dict) else None
    return [x for x in (lst or []) if isinstance(x, dict) and x.get("decision") in ("REDACT", "KEEP")]


def _row_key(r):
    return (r.get("doc_id"), r.get("mode"), r.get("prompt_hash"), r.get("model_digest"), r.get("window"), r.get("overlap"),
            r.get("pipeline_version"))


def load_rows(path, meta, mode, window, overlap):
    """Rows already on disk that were produced with the same prompt hash, model digest and window settings."""
    done = {}
    if os.path.exists(path):
        for r in load_jsonl(path):
            if _row_key(r) == (r.get("doc_id"), mode, meta["prompt_hash"], meta.get("model_digest"), window, overlap, PIPELINE_VERSION):
                done[r["doc_id"]] = r
    return done


STAMP_KEYS = ("prompt_hash", "model_digest", "git_commit", "git_dirty", "prompt_version", "machine", "prompt_modified")


def run_detection(docs, out_path, chat_fn, meta, mode="standalone", candidates_by_doc=None,
                  window=1200, overlap=100, log=print, max_seconds=None, clock=time.perf_counter):
    """Resume-safe: one JSONL row per doc appended to out_path.
    meta: prompt_hash, model_digest, model_tag, git_*, prompt_version, machine, prompt_modified.
    max_seconds: stop starting NEW docs once this wall-clock budget is used up (finished rows stay; re-run resumes)."""
    os.makedirs(os.path.dirname(os.path.abspath(out_path)), exist_ok=True)
    done = load_rows(out_path, meta, mode, window, overlap)
    todo = [d for d in docs if d["doc_id"] not in done]
    fp = sample_fingerprint(docs)
    left = [len(make_windows(d["text"], window, overlap)) for d in todo]
    log(f"{len(done)} docs already done, {len(todo)} to run -> {out_path}")
    t0, wlat = clock(), []
    for i, d in enumerate(todo, 1):
        if max_seconds is not None and clock() - t0 >= max_seconds:
            log(f"time budget reached: {len(done)}/{len(docs)} docs completed ({i - 1} this session). Re-run to resume.")
            break
        row = detect_doc(d, chat_fn, window, overlap, mode, (candidates_by_doc or {}).get(d["doc_id"]))
        if row["n_invalid"] + row["n_truncated"] == row["n_windows"] and any("ConnectionRefused" in k for k in row["errors"]):
            raise RuntimeError("Ollama refused the connection: is `ollama serve` running? (no row written)")
        row.update({k: meta.get(k) for k in STAMP_KEYS})
        row["prompt_modified"] = bool(meta.get("prompt_modified", False))
        row.update({"model_tag": meta.get("model_tag"), "window": window, "overlap": overlap, "sample_fingerprint": fp,
                    "pipeline_version": PIPELINE_VERSION})
        with open(out_path, "a", encoding="utf-8", newline="\n") as f:
            f.write(json.dumps(row) + "\n")
        done[d["doc_id"]] = row
        wlat += [w["latency_s"] for w in row["windows"] if w["latency_s"] is not None]
        log(f"[{i}/{len(todo)}] {d['doc_id']} {d.get('source')} win={row['n_windows']} trunc={row['n_truncated']} "
            f"loop={row['n_loop']} dense={row['n_dense']} split={row['n_split']} "
            f"inv={row['n_invalid']} unm={row['n_unmatched']} preds={len(row['preds'])} {row['latency_s']:.1f}s")
        if i % 10 == 0 and wlat:
            med, rest = float(np.median(wlat)), sum(left[i:])
            log(f"   ETA ~{med * rest / 60:.1f} min ({rest} windows left x median {med:.1f}s/window)")
    log(f"{len(done)}/{len(docs)} docs completed")
    return [done[d["doc_id"]] for d in docs if d["doc_id"] in done]

def recovery_counts(rows):
    """Per source: windows, loop-aborted, dense-truncated, salvaged objects, split-recovered, still-failed, unmatched strings."""
    out = {}
    for r in rows:
        a = out.setdefault(r.get("source"), {"windows": 0, "loop": 0, "dense": 0, "salvaged": 0, "recovered": 0, "failed": 0,
                                             "unmatched": 0, "returned": 0, "samples": []})
        for w in r.get("windows", []):
            a["salvaged"] += w.get("n_spans_salvaged", 0)
            if w.get("half"):
                continue
            a["windows"] += 1
            cause = w.get("cause", "none")
            a["loop"] += cause == "loop"
            a["dense"] += cause == "dense"
            if cause != "none" or w.get("truncated"):
                a["failed" if w.get("failed_after_split") else "recovered"] += 1
            a["unmatched"] += len(w.get("unmatched", []))
            a["returned"] += w.get("n_returned", 0)
            a["samples"] += [u for u in w.get("unmatched", []) if len(a["samples"]) < 10]
    return out


def print_recovery_report(rows, log=print):
    for src, a in sorted(recovery_counts(rows).items(), key=lambda kv: str(kv[0])):
        log(f"{src}: windows={a['windows']} loop-aborted={a['loop']} dense-truncated={a['dense']} salvaged={a['salvaged']} "
            f"split-recovered={a['recovered']} still-failed={a['failed']}")
        log(f"   unmatched strings: {a['unmatched']} ({a['unmatched'] / max(a['returned'], 1):.3f} per returned string)")
        for u in a["samples"]:
            log(f"      {u!r}")
    top = sorted(rows, key=lambda r: -r.get("n_unmatched", 0))[:5]
    log("top documents by unmatched strings: " + ", ".join(f"{r['doc_id']} ({r['source']}): {r.get('n_unmatched', 0)}" for r in top))


### Gold adapters

In [ ]:
def _span(s, decision, label=None):
    return {"start": s["start"], "end": s["end"], "label": label or s.get("label"), "decision": decision}


def gold_crapii(doc):
    """CRAPII: every gold span is redact."""
    return [_span(s, "redact") for s in doc["spans"]]


def gold_tab(doc, scope="direct+quasi", annotator="first"):
    """TAB. scope 'direct+quasi': DIRECT, QUASI = redact; NO_MASK = keep.
    scope 'direct': DIRECT = redact, QUASI = ignore, NO_MASK = keep. MISC/QUANTITY (ignored_spans) = ignore.
    annotator 'first' = primary spans only; 'union' = primary plus every other annotator."""
    spans = list(doc["spans"])
    if annotator == "union":
        for lst in (doc.get("other_annotations") or {}).values():
            spans += lst
    out = []
    for s in spans:
        it = (s.get("attrs") or {}).get("identifier_type")
        if s.get("label") not in UNIFIED_LABELS:
            out.append(_span(s, "ignore", s.get("label")))
        elif it == "NO_MASK":
            out.append(_span(s, "keep"))
        elif it == "QUASI" and scope == "direct":
            out.append(_span(s, "ignore"))
        else:
            out.append(_span(s, "redact"))
    out += [_span(s, "ignore", s.get("label")) for s in doc.get("ignored_spans") or []]
    return out


def expected_decision(label):
    return "KEEP" if label in P5_KEEP else "REDACT"


def gold_synthetic(case):
    """Synthetic system-check cases: spans carry expected_decision."""
    return [_span(s, "redact" if s["expected_decision"] == "REDACT" else "keep", s.get("gold_label"))
            for s in case["spans"]]


def gold_for(doc, scope="direct+quasi", annotator="first"):
    src = doc.get("source")
    if src == "tab":
        return gold_tab(doc, scope, annotator)
    if src == "synthetic":
        return gold_synthetic(doc)
    return gold_crapii(doc)

### Metrics, bootstrap CIs, error examples

In [ ]:
def _mask(n, spans, nz):
    m = np.zeros(n, bool)
    for s, e in spans:
        m[max(s, 0):min(e, n)] = True
    return m & nz


def _dedupe(gold):
    seen, out = set(), []
    for g in gold:
        k = (g["start"], g["end"], g["label"], g["decision"])
        if k not in seen:
            seen.add(k)
            out.append(g)
    return out


def doc_counts(text, gold, preds):
    """Raw counts for one doc (whitespace excluded everywhere). Priority of gold regions: redact > keep > ignore.
    Only REDACT predictions count (missing decision = REDACT); predictions fully inside ignore regions are dropped."""
    n = len(text)
    nz = np.fromiter((not c.isspace() for c in text), bool, n)
    gold = _dedupe(gold)
    red = [g for g in gold if g["decision"] == "redact"]
    keep = [g for g in gold if g["decision"] == "keep"]
    R = _mask(n, [(g["start"], g["end"]) for g in red], nz)
    K = _mask(n, [(g["start"], g["end"]) for g in keep], nz) & ~R
    I = _mask(n, [(g["start"], g["end"]) for g in gold if g["decision"] == "ignore"], nz) & ~R & ~K
    redp = [p for p in preds if p.get("decision", "REDACT") == "REDACT" and p["end"] > p["start"]]
    C = _mask(n, [(p["start"], p["end"]) for p in redp], nz)          # all REDACT chars: used for coverage
    kept = []
    for p in redp:
        s, e = max(p["start"], 0), min(p["end"], n)
        seg = nz[s:e] & ~I[s:e]
        if seg.any():                                                   # not whitespace-only, not fully ignored
            kept.append((s, e, not (seg & ~R[s:e]).any()))
    F = _mask(n, [(s, e) for s, e, _ in kept], nz) & ~I
    c = {"n_docs": 1, "pred_n": len(kept), "pred_tp": sum(t for _, _, t in kept),
         "flag_chars": int(F.sum()), "flag_nonpii": int((F & ~R).sum()),
         "nonpii_chars": int((nz & ~R & ~I).sum()), "gold_chars": int(R.sum()), "gold_uncov": int((R & ~C).sum())}
    rec_n = rec_c = 0
    for g in red:
        gm = _mask(n, [(g["start"], g["end"])], nz)
        if not gm.any():
            continue
        cov = not (gm & ~C).any()
        rec_n += 1
        rec_c += cov
        c[f"rec_n|{g['label']}"] = c.get(f"rec_n|{g['label']}", 0) + 1
        c[f"rec_c|{g['label']}"] = c.get(f"rec_c|{g['label']}", 0) + cov
    keep_n = keep_hit = 0
    for g in keep:
        gm = _mask(n, [(g["start"], g["end"])], nz) & ~R
        if not gm.any():
            continue
        hit = bool((gm & C).any())
        keep_n += 1
        keep_hit += hit
        c[f"keep_n|{g['label']}"] = c.get(f"keep_n|{g['label']}", 0) + 1
        c[f"keep_hit|{g['label']}"] = c.get(f"keep_hit|{g['label']}", 0) + hit
    has_gold = rec_n > 0
    c.update({"rec_n": rec_n, "rec_c": rec_c, "keep_n": keep_n, "keep_hit": keep_hit,
              "docs_gold": int(has_gold), "docs_leaked": int(has_gold and rec_c < rec_n),
              "docs_nogold": int(not has_gold), "docs_false_alarm": int((not has_gold) and len(kept) > 0)})
    return c


def _div(a, b):
    return float(a) / b if b else float("nan")


def metrics_from_sums(s):
    """Aggregate metrics from summed doc_counts (a dict; absent keys = 0)."""
    g = lambda k: float(s.get(k, 0))
    P, R = _div(g("pred_tp"), g("pred_n")), _div(g("rec_c"), g("rec_n"))
    p, r = (0.0 if P != P else P), (0.0 if R != R else R)
    m = {"span_recall": R, "span_precision": P,
         "f2": _div(5 * p * r, 4 * p + r),
         "char_over_redaction_pct": 100 * _div(g("flag_nonpii"), g("flag_chars")),
         "non_pii_flag_rate": _div(g("flag_nonpii"), g("nonpii_chars")),
         "char_leakage_pct": 100 * _div(g("gold_uncov"), g("gold_chars")),
         "doc_leakage": _div(g("docs_leaked"), g("docs_gold")),
         "doc_false_alarm": _div(g("docs_false_alarm"), g("docs_nogold")),
         "keep_false_redaction_rate": _div(g("keep_hit"), g("keep_n"))}
    for k in s:
        if k.startswith("rec_n|"):
            lab = k.split("|", 1)[1]
            m[f"recall|{lab}"] = _div(g(f"rec_c|{lab}"), g(k))
        elif k.startswith("keep_n|"):
            lab = k.split("|", 1)[1]
            m[f"keep_false|{lab}"] = _div(g(f"keep_hit|{lab}"), g(k))
    return m


def bootstrap_ci(counts, n_boot=1000, seed=42):
    """95% percentile CIs over documents. counts: DataFrame, one row per doc."""
    cols, M = list(counts.columns), counts.to_numpy(float)
    rng = np.random.default_rng(seed)
    N = len(M)
    reps = [metrics_from_sums(dict(zip(cols, M[rng.integers(0, N, N)].sum(0)))) for _ in range(n_boot)]
    ci = {}
    for k in reps[0]:
        v = np.array([r[k] for r in reps])
        v = v[~np.isnan(v)]
        ci[k] = tuple(np.percentile(v, [2.5, 97.5])) if len(v) else (float("nan"), float("nan"))
    return ci


HEADLINE = ["span_recall", "span_precision", "f2", "char_over_redaction_pct", "non_pii_flag_rate",
            "char_leakage_pct", "doc_leakage", "doc_false_alarm", "keep_false_redaction_rate"]


def score_table(rows, docs_by_id, scope="direct+quasi", annotator="first", sources=None, n_boot=1000, seed=42):
    """Per (source, split) metrics with 95% CIs. Returns (summary_df, label_df). Sources are never pooled."""
    groups = {}
    for r in rows:
        d = docs_by_id.get(r["doc_id"])
        if d is None or (sources and d["source"] not in sources):
            continue
        groups.setdefault((d["source"], d.get("split")), []).append(
            doc_counts(d["text"], gold_for(d, scope, annotator), r["preds"]))
    summ, lab = [], []
    for (src, sp), cl in sorted(groups.items(), key=lambda kv: (str(kv[0][0]), str(kv[0][1]))):
        df = pd.DataFrame(cl).fillna(0)
        m, ci = metrics_from_sums(df.sum().to_dict()), bootstrap_ci(df, n_boot, seed)
        base = {"source": src, "split": sp, "scope": scope if src == "tab" else "n/a",
                "annotator": annotator if src == "tab" else "n/a"}
        row = {**base, "n_docs": len(df), "n_gold_redact": int(df.get("rec_n", pd.Series([0])).sum()),
               "n_gold_keep": int(df.get("keep_n", pd.Series([0])).sum())}
        for k in HEADLINE:
            row[k], row[k + "_lo"], row[k + "_hi"] = m[k], ci[k][0], ci[k][1]
        summ.append(row)
        for k, v in m.items():
            if "|" in k:
                kind, label = k.split("|", 1)
                nk = ("rec_n|" if kind == "recall" else "keep_n|") + label
                lab.append({**base, "kind": kind, "label": label, "n": int(df[nk].sum()), "value": v,
                            "lo": ci[k][0], "hi": ci[k][1], "n_docs": len(df)})
    return pd.DataFrame(summ), pd.DataFrame(lab)


def fmt_ci(df, cols=HEADLINE):
    """Display copy: 'value [lo, hi]' strings."""
    out = df[["source", "split", "scope", "n_docs", "n_gold_redact", "n_gold_keep"]].copy()
    for k in cols:
        out[k] = [f"{v:.3f} [{lo:.3f}, {hi:.3f}]" for v, lo, hi in zip(df[k], df[k + "_lo"], df[k + "_hi"])]
    return out


def context(text, s, e, width=120):
    if e - s >= width:
        return text[s:s + width]
    pad = (width - (e - s)) // 2
    return text[max(0, s - pad):e + pad][:width]


def _error_lists(rows, docs_by_id, scope, annotator):
    miss, fals = [], []
    for r in rows:
        d = docs_by_id.get(r["doc_id"])
        if d is None:
            continue
        text, n = d["text"], len(d["text"])
        nz = np.fromiter((not c.isspace() for c in text), bool, n)
        gold = _dedupe(gold_for(d, scope, annotator))
        redp = [(p["start"], p["end"]) for p in r["preds"] if p.get("decision", "REDACT") == "REDACT"]
        C = _mask(n, redp, nz)
        R = _mask(n, [(g["start"], g["end"]) for g in gold if g["decision"] == "redact"], nz)
        for g in gold:
            gm = _mask(n, [(g["start"], g["end"])], nz)
            base = {"source": d["source"], "split": d.get("split"), "doc_id": d["doc_id"], "label": g["label"],
                    "text": text[g["start"]:g["end"]][:120], "context": context(text, g["start"], g["end"])}
            if g["decision"] == "redact" and gm.any() and (gm & ~C).any():
                miss.append({**base, "uncovered_chars": int((gm & ~C).sum())})
            elif g["decision"] == "keep" and (gm & ~R).any() and (gm & ~R & C).any():
                fals.append({**base, "overlap_chars": int((gm & ~R & C).sum())})
    return miss, fals


def worst_errors(rows, docs_by_id, scope="direct+quasi", annotator="first", k=10):
    """Worst-k per source: gold redact spans missed (most uncovered chars) and keep spans falsely redacted."""
    miss, fals = _error_lists(rows, docs_by_id, scope, annotator)

    def top(lst, key):
        df = pd.DataFrame(lst)
        if df.empty:
            return df
        return df.sort_values(key, ascending=False).groupby("source", sort=False).head(k).reset_index(drop=True)
    return top(miss, "uncovered_chars"), top(fals, "overlap_chars")


def error_counts(rows, docs_by_id, scope="direct+quasi", annotator="first"):
    """Aggregate error mining (no text): counts of misses and false redactions by source, split and gold label."""
    miss, fals = _error_lists(rows, docs_by_id, scope, annotator)
    parts = [pd.DataFrame(lst).groupby(["source", "split", "label"]).size().rename("count").reset_index().assign(kind=kind)
             for kind, lst in (("miss", miss), ("false_redaction", fals)) if lst]
    cols = ["kind", "source", "split", "label", "count"]
    return pd.concat(parts)[cols].reset_index(drop=True) if parts else pd.DataFrame(columns=cols)


def load_baseline(path):
    """Baseline predictions JSONL {doc_id, spans:[{start,end,text,label,decision?}]} -> rows (missing decision = REDACT)."""
    rows = []
    for r in load_jsonl(path):
        spans = [{**s, "decision": s.get("decision", "REDACT")} for s in r["spans"]]
        rows.append({"doc_id": r["doc_id"], "preds": spans})
    return rows

### System-check, split guard, prompt guard, sampling, fingerprint

In [ ]:
def decision_agreement(gold_spans, preds):
    """Decision-level agreement with expected_decision. gold_spans: [{start,end,expected_decision}].
    A gold span is predicted REDACT if any REDACT prediction overlaps it, otherwise KEEP."""
    red = [(p["start"], p["end"]) for p in preds if p.get("decision", "REDACT") == "REDACT"]
    agree, bad = 0, []
    for g in gold_spans:
        got = "REDACT" if any(s < g["end"] and g["start"] < e for s, e in red) else "KEEP"
        if got == g["expected_decision"]:
            agree += 1
        else:
            bad.append({"text": g.get("text"), "expected": g["expected_decision"], "got": got})
    return agree, len(gold_spans), bad


def pred_set(preds):
    return sorted({(p["text"], p["decision"]) for p in preds})


def set_agreement(a, b):
    """Jaccard agreement (%) of two (text, decision) sets; both empty = 100."""
    a, b = {tuple(x) for x in a}, {tuple(x) for x in b}
    return 100.0 * len(a & b) / len(a | b) if (a | b) else 100.0


def timing_medians(windows):
    """Medians over windows of the server-reported timing fields; nan for fields that are absent (old files).
    overhead_s = latency - generation - prompt eval (HTTP, JSON schema setup, scheduling, anything not measured by the server)."""
    med = lambda xs: float(np.median(xs)) if xs else float("nan")
    got = lambda k: [w[k] for w in windows if w.get(k) is not None]
    over = [w["latency_s"] - w["gen_s"] - w["prompt_eval_s"] for w in windows
            if None not in (w.get("latency_s"), w.get("gen_s"), w.get("prompt_eval_s"))]
    return {"median_gen_s": med(got("gen_s")), "median_prompt_eval_s": med(got("prompt_eval_s")),
            "median_overhead_s": med(over), "median_load_s": med(got("load_s")), "median_total_s": med(got("total_s")),
            "median_prompt_eval_count": med(got("prompt_eval_count"))}


def latency_stats(rows):
    """Stats over all windows of rows in run order, excluding the very first call (it includes model load)."""
    flat = [w for r in rows for w in r["windows"]]
    if not flat:
        return {}
    first, rest = flat[0], flat[1:]
    med = lambda xs: float(np.median(xs)) if xs else float("nan")
    return {"n_windows": len(flat), "first_call_latency_s": first["latency_s"], "model_load_s": first.get("load_s"),
            "median_latency_s": med([w["latency_s"] for w in rest if w["latency_s"] is not None]),
            "median_tokens_per_s": med([w["tokens_per_s"] for w in rest if w["tokens_per_s"]]),
            **timing_medians(rest),
            "valid_json_rate": (sum(w["valid"] for w in rest) / len(rest)) if rest else float("nan"),
            "n_truncated": sum(w["truncated"] for w in rest)}


def project_runtime(docs, median_s, window=1200, overlap=100):
    n = sum(len(make_windows(d["text"], window, overlap)) for d in docs)
    return {"n_docs": len(docs), "n_windows": n, "est_seconds": n * median_s}


SPLIT_ORDER = ("dev", "train", "test")


def check_run_splits(splits, allow_test=False):
    """Validate RUN_SPLITS (dev always before train). 'test' stays locked unless ALLOW_TEST is True."""
    splits = list(splits)
    if "test" in splits and allow_test is not True:
        raise RuntimeError("test is locked until the pipeline is frozen: remove 'test' from RUN_SPLITS "
                           "(ALLOW_TEST must stay False until then).")
    bad = [s for s in splits if s not in SPLIT_ORDER]
    if bad:
        raise ValueError(f"unknown split(s) {bad}; allowed: {SPLIT_ORDER}")
    return sorted(dict.fromkeys(splits), key=SPLIT_ORDER.index)


def enforce_prompt_hash(actual, expected, allow_change=False, log=print):
    """Returns prompt_modified (bool). A mismatch raises unless allow_change, which prints a loud warning."""
    if actual == expected:
        return False
    msg = f"PROMPT HASH MISMATCH: expected {expected}, got {actual}: the system prompt or schema was edited."
    if not allow_change:
        raise RuntimeError(msg + " Set ALLOW_PROMPT_CHANGE=True to run anyway (rows are then stamped prompt_modified=true).")
    log("!" * 78)
    log("WARNING: " + msg)
    log("ALLOW_PROMPT_CHANGE=True: every row is stamped prompt_modified=true; results are NOT comparable to the frozen prompt.")
    log("!" * 78)
    return True


def machine_stamp(hw, ollama_ver):
    """What every row records about the machine (no hostname, username or paths)."""
    return {"chip": hw.get("chip"), "ram_gb": hw.get("ram_gb"), "os": hw.get("os"),
            "gpu": (hw.get("gpu") or {}).get("name"), "label": hw.get("label"), "ollama_version": ollama_ver}


def sample_order(docs, stratified=False, seed=42):
    """Deterministic order whose first n docs are the sample of size n (prefix property).
    stratified (CRAPII): two seeded permutations (docs with >= 1 gold span, docs with none), interleaved 1:1."""
    pool = sorted(docs, key=lambda d: d["doc_id"])

    def perm(xs, tag):
        xs = list(xs)
        random.Random(f"{seed}:{tag}").shuffle(xs)
        return xs
    if not stratified:
        return perm(pool, "all")
    pos, neg = perm([d for d in pool if d["spans"]], "pos"), perm([d for d in pool if not d["spans"]], "neg")
    out = []
    for i in range(max(len(pos), len(neg))):
        out += pos[i:i + 1] + neg[i:i + 1]
    return out


def sample_docs(docs, n, stratified=False, seed=42):
    return sample_order(docs, stratified, seed)[:n]


def interleave_sources(by_source):
    """Merge per-source lists proportionally to their lengths (deterministic), so an early stop keeps every source."""
    keyed = [((k + 0.5) / len(lst), si, k, d) for si, lst in enumerate(by_source.values()) for k, d in enumerate(lst)]
    keyed.sort(key=lambda t: t[:3])
    return [t[3] for t in keyed]


def select_split(docs, sizes, seed=42, ids=None):
    """Docs of one split: sizes = {source: n}. ids (list of doc_ids) overrides sampling."""
    by = {}
    for src, n in sizes.items():
        pool = [d for d in docs if d["source"] == src]
        if ids is not None:
            want = set(ids)
            by[src] = [d for d in sorted(pool, key=lambda d: d["doc_id"]) if d["doc_id"] in want]
        else:
            by[src] = sample_docs(pool, n, src == "crapii", seed)
    return interleave_sources({k: v for k, v in by.items() if v})


def sample_fingerprint(docs):
    return hashlib.sha1("\n".join(sorted(d["doc_id"] for d in docs)).encode()).hexdigest()[:12]


TEXT_SHA1_DOCS = 200


def text_sha1(texts):
    """sha1 of the UTF-8 encoding of the decoded texts joined by "\\n" (catches decoding/line-ending differences between machines)."""
    return hashlib.sha1("\n".join(texts).encode("utf-8")).hexdigest()


def data_fingerprint(data_dir, splits=SPLIT_ORDER, parse=("dev", "train")):
    """sha256 of each file's bytes. Splits not in `parse` (test) are only hashed and line-counted, never JSON-parsed.
    Parsed splits also get text_sha1: of the `text` of the first 200 docs in file order, after JSON parsing."""
    fp = {}
    for sp in splits:
        path = os.path.join(data_dir, f"{sp}.jsonl")
        h, rows, per, texts = hashlib.sha256(), 0, ({} if sp in parse else None), []
        with open(path, "rb") as f:
            for line in f:
                h.update(line)
                if line.strip():
                    rows += 1
                    if per is not None:
                        doc = json.loads(line.decode("utf-8"))
                        src = doc.get("source")
                        per[src] = per.get(src, 0) + 1
                        if len(texts) < TEXT_SHA1_DOCS:
                            texts.append(doc.get("text") or "")
        fp[f"{sp}.jsonl"] = {"sha256": h.hexdigest(), "rows": rows, "docs_per_source": per,
                             "text_sha1": text_sha1(texts) if per is not None else None}
    return fp


def compare_fingerprint(fp, ref):
    """Warnings when this machine's data differs from the reference fingerprint (text_sha1 only compared when both have it)."""
    warns = []
    for name, v in fp.items():
        r = ref.get(name) or {}
        if r.get("sha256") != v["sha256"]:
            warns.append(f"{name} differs from llm_p5/reference/data_fingerprint.json: you may not have the shared data version.")
        if r.get("text_sha1") and v.get("text_sha1") and r["text_sha1"] != v["text_sha1"]:
            warns.append(f"{name}: decoded-text fingerprint (text_sha1) differs from the reference: the text decodes differently "
                         "on this machine (encoding or line-ending conversion of the data files?).")
    return warns

### Machine comparison helpers

In [ ]:
def latest_rows(rows):
    """Last row per doc_id (a results file can hold several attempts)."""
    d = {}
    for r in rows:
        d[r["doc_id"]] = r
    return list(d.values())


def check_comparable(rows_a, rows_b):
    """Warnings for every mismatch; valid=False when the model digest differs or nothing is shared."""
    warns = []
    for f in ("prompt_hash", "prompt_version", "model_digest", "window", "overlap"):
        va, vb = {r.get(f) for r in rows_a}, {r.get(f) for r in rows_b}
        for tag, v in (("A (this machine)", va), ("B (other)", vb)):
            if len(v) > 1:
                warns.append(f"file {tag} mixes several {f} values: {sorted(map(str, v))}")
        if va != vb:
            warns.append(f"{f} differs: {sorted(map(str, va))} vs {sorted(map(str, vb))}")
    da, db = {r.get("model_digest") for r in rows_a}, {r.get("model_digest") for r in rows_b}
    ia, ib = {r["doc_id"] for r in rows_a}, {r["doc_id"] for r in rows_b}
    common = sorted(ia & ib)
    if ia != ib:
        warns.append(f"doc coverage differs: A={len(ia)}, B={len(ib)}, common={len(common)}, only A={len(ia - ib)}, only B={len(ib - ia)}")
    if any(r.get("prompt_modified") for r in rows_a + rows_b):
        warns.append("some rows are stamped prompt_modified=true")
    digest_ok = len(da) == 1 and da == db and None not in da
    if not digest_ok:
        warns.append("model digest mismatch or missing: COMPARISON INVALID")
    return {"warnings": warns, "valid": bool(digest_ok and common), "common": common}


def run_agreement(rows_a, rows_b):
    """Per-doc agreement on common docs: Jaccard of (text, label, decision) sets, identical-prediction flag,
    Jaccard of REDACT text sets. Returns (summary dict, per-doc DataFrame)."""
    A, B = {r["doc_id"]: r for r in rows_a}, {r["doc_id"]: r for r in rows_b}
    jac = lambda x, y: len(x & y) / len(x | y) if (x | y) else 1.0
    per = []
    for i in sorted(set(A) & set(B)):
        sa = {(p["text"], p.get("label"), p["decision"]) for p in A[i]["preds"]}
        sb = {(p["text"], p.get("label"), p["decision"]) for p in B[i]["preds"]}
        ra = {p["text"] for p in A[i]["preds"] if p["decision"] == "REDACT"}
        rb = {p["text"] for p in B[i]["preds"] if p["decision"] == "REDACT"}
        per.append({"doc_id": i, "source": A[i].get("source"), "jaccard_all": jac(sa, sb),
                    "jaccard_redact": jac(ra, rb), "identical": sa == sb})
    df = pd.DataFrame(per, columns=["doc_id", "source", "jaccard_all", "jaccard_redact", "identical"])
    summ = {"n_common": len(df), "mean_jaccard_all": float(df.jaccard_all.mean()) if len(df) else float("nan"),
            "share_identical": float(df.identical.mean()) if len(df) else float("nan"),
            "mean_jaccard_redact": float(df.jaccard_redact.mean()) if len(df) else float("nan")}
    return summ, df


def speed_stats(rows, gpu=None):
    """Speed and health of one machine's run (first call excluded from medians: it includes model load)."""
    st = latency_stats(rows)
    wins = [w for r in rows for w in r["windows"]]
    mach = (rows[-1].get("machine") or {}) if rows else {}
    return {"chip": mach.get("chip"), "ram_gb": mach.get("ram_gb"), "n_docs": len(rows), "n_windows": len(wins),
            "median_s_per_window": st.get("median_latency_s"), "median_tokens_per_s": st.get("median_tokens_per_s"),
            "median_gen_s": st.get("median_gen_s"), "median_prompt_eval_s": st.get("median_prompt_eval_s"),
            "median_overhead_s": st.get("median_overhead_s"), "gpu": ((mach.get("gpu") or {}).get("name") if isinstance(mach.get("gpu"), dict) else mach.get("gpu")),
            "total_wall_s": sum(r["latency_s"] for r in rows), "gpu_share": gpu,
            "n_truncated": sum(r["n_truncated"] for r in rows), "n_unmatched": sum(r["n_unmatched"] for r in rows),
            "valid_rate": (sum(w["valid"] for w in wins) / len(wins)) if wins else float("nan")}


DIFF_METRICS = ["span_recall", "span_precision", "f2", "char_leakage_pct", "keep_false_redaction_rate"]


def metrics_diff(sa, sb):
    """Side-by-side metrics of two score_table summaries (A = this machine, B = other) with A - B."""
    m = sa.merge(sb, on=["source", "split", "scope"], suffixes=("_a", "_b"))
    out = m[["source", "split", "scope", "n_docs_a"]].rename(columns={"n_docs_a": "n_docs"})
    for k in DIFF_METRICS:
        out[k + "_a"], out[k + "_b"], out[k + "_diff"] = m[k + "_a"], m[k + "_b"], m[k + "_a"] - m[k + "_b"]
    return out.reset_index(drop=True)

## Data check

In [ ]:
RUN_SPLITS = check_run_splits(RUN_SPLITS, ALLOW_TEST)       # raises if "test" is requested while locked
need = [DATA_DIR / f"{sp}.jsonl" for sp in SPLIT_ORDER]
missing = [str(p.relative_to(ROOT)) for p in need if not p.exists()]
assert not missing, (f"Missing {missing}. Unzip the shared data folder so that dev.jsonl, train.jsonl and test.jsonl exist in "
                     f"{DATA_DIR.relative_to(ROOT)}/ (the data is gitignored and comes from the team zip).")
FINGERPRINT = data_fingerprint(DATA_DIR)                    # test is only hashed, never parsed
for name, fp in FINGERPRINT.items():
    docs_txt = fp["docs_per_source"] if fp["docs_per_source"] is not None else "not parsed (locked)"
    print(f"{name}: rows={fp['rows']} docs_per_source={docs_txt}\n   sha256={fp['sha256']}\n   text_sha1={fp['text_sha1']}")
ref_fp = REFERENCE_DIR / "data_fingerprint.json"
if ref_fp.exists():
    for w in compare_fingerprint(FINGERPRINT, json.loads(ref_fp.read_text(encoding="utf-8"))):
        print("WARNING:", w)
RESULTS_DIR.mkdir(exist_ok=True); INCOMING_DIR.mkdir(exist_ok=True)
(RESULTS_DIR / "data_fingerprint.json").write_text(json.dumps(FINGERPRINT, indent=1), encoding="utf-8", newline="\n")

In [ ]:
DOCS = {sp: load_jsonl(DATA_DIR / f"{sp}.jsonl") for sp in RUN_SPLITS}
ids = json.loads(Path(SAMPLE_IDS_FILE).read_text(encoding="utf-8")) if SAMPLE_IDS_FILE else None
SELECTED = {sp: select_split(DOCS[sp], SAMPLE_SIZES[sp], SEED, ids) for sp in RUN_SPLITS}
SAMPLE_FP = {sp: sample_fingerprint(SELECTED[sp]) for sp in RUN_SPLITS}
DOCS_BY_ID = {d["doc_id"]: d for ds in SELECTED.values() for d in ds}
del DOCS
for sp, ds in SELECTED.items():
    comp = {src: (sum(d["source"] == src for d in ds), sum(d["source"] == src and bool(d["spans"]) for d in ds))
            for src in sorted({d["source"] for d in ds})}
    print(f"{sp}: {len(ds)} docs; per source (docs, with gold): {comp}; sample_fingerprint={SAMPLE_FP[sp]}")

## System check
30 fictional cases (committed, safe to share) through the same detector. The prompt-hash guard runs first. First call excluded from
stats (it includes model load). Needs Ollama running and the model pulled.

In [ ]:
PROMPT_MODIFIED = enforce_prompt_hash(prompt_hash("standalone"), EXPECTED_PROMPT_HASH, ALLOW_PROMPT_CHANGE)
HW = {**hw_info(HARDWARE_OVERRIDE), "label": MACHINE_LABEL}
MACHINE_TAG = machine_tag(HW, MACHINE_LABEL)
INFO = model_info(MODEL_TAG, OLLAMA_URL)
OLLAMA_VER = ollama_version(OLLAMA_URL)
assert INFO["digest"], f"{MODEL_TAG} not found: start Ollama and run `ollama pull {MODEL_TAG}`."
PHASH = prompt_hash("standalone")
MACHINE = machine_stamp(HW, OLLAMA_VER)
META = {"prompt_hash": PHASH, "model_digest": INFO["digest"], "model_tag": MODEL_TAG, "prompt_version": PROMPT_VERSION,
        "machine": MACHINE, "prompt_modified": PROMPT_MODIFIED, **git_info()}
chat_fn = make_chat_fn(MODEL_TAG, OLLAMA_URL, NUM_CTX, NUM_PREDICT, DEADLINE, SEED)
print(f"prompt {PROMPT_VERSION} hash {PHASH}; model digest {INFO['digest'][:12]}; sample fingerprints {SAMPLE_FP}")

cases = load_jsonl(CASES_PATH)
chk_docs = [{"doc_id": c["case_id"], "source": "synthetic", "split": "check", "text": c["original"], "spans": c["spans"]}
            for c in cases]
chk_path = RESULTS_DIR / "system_check_rows.jsonl"
chk_path.unlink(missing_ok=True)                       # always a fresh timing run
chk_rows = run_detection(chk_docs, chk_path, chat_fn, META, "standalone", None, WINDOW, OVERLAP)
GPU = gpu_share(MODEL_TAG, OLLAMA_URL)

STATS = latency_stats(chk_rows)
per_case, ag, tot_ = {}, 0, 0
for d, r in zip(chk_docs, chk_rows):
    a, t, bad = decision_agreement(d["spans"], r["preds"])
    ag, tot_ = ag + a, tot_ + t
    per_case[d["doc_id"]] = {"agreement": a / t if t else None, "mismatches": bad, "pred_set": pred_set(r["preds"])}
STATS["decision_agreement"] = ag / tot_ if tot_ else float("nan")
print(f"hardware: {HW['chip']}, {HW['ram_gb']} GB, GPU {HW['gpu']}; machine tag {MACHINE_TAG}")
print(f"model load (first call): {STATS['model_load_s'] or float('nan'):.1f}s, first call total {STATS['first_call_latency_s']:.1f}s (excluded from stats)")
print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in STATS.items()})

CHECK_PATH = RESULTS_DIR / f"system_check_{MACHINE_TAG}.json"
REPORT = {"hardware": HW, "ollama_version": OLLAMA_VER, "model": {"tag": MODEL_TAG, **INFO}, "gpu_share": GPU,
          "prompt_hash": PHASH, "prompt_version": PROMPT_VERSION, "prompt_modified": PROMPT_MODIFIED,
          "sample_fingerprint": SAMPLE_FP, "config": {"window": WINDOW, "overlap": OVERLAP, "num_ctx": NUM_CTX,
          "num_predict": NUM_PREDICT, "deadline_s": DEADLINE, "seed": SEED},
          "stats": STATS, "cases": per_case, "data_fingerprint": FINGERPRINT}
CHECK_PATH.write_text(json.dumps(REPORT, indent=1), encoding="utf-8", newline="\n")
print("wrote", CHECK_PATH.relative_to(ROOT), "(send this file first)")

In [ ]:
ref_path = REFERENCE_DIR / "system_check_reference.json"
if ref_path.exists():
    REF = json.loads(ref_path.read_text(encoding="utf-8"))
    rs = REF["stats"]
    tbl = pd.DataFrame({
        "this machine": [STATS["median_latency_s"], STATS["median_gen_s"], STATS["median_prompt_eval_s"], STATS["median_overhead_s"],
                         STATS["median_tokens_per_s"], GPU, STATS["valid_json_rate"]],
        "reference": [rs.get("median_latency_s"), rs.get("median_gen_s"), rs.get("median_prompt_eval_s"), rs.get("median_overhead_s"),
                      rs.get("median_tokens_per_s"), REF.get("gpu_share"), rs.get("valid_json_rate")],
    }, index=["median latency / window (s)", "median generation (s)", "median prompt eval (s)", "median overhead (s)",
              "tokens / s", "GPU share", "valid JSON rate"]).astype(float)   # fields missing in an older reference show as nan
    print(tbl.round(3).to_string())
    print("digest match:", INFO["digest"] == REF["model"]["digest"])
    if INFO["digest"] != REF["model"]["digest"]:
        print("WARNING: model digest differs from the reference: results are not comparable. Re-pull the exact model.")
    if REF.get("prompt_hash") != PHASH:
        print("WARNING: prompt hash differs from the reference run.")
    agr = {cid: set_agreement(per_case[cid]["pred_set"], REF["cases"][cid]["pred_set"])
           for cid in per_case if cid in REF.get("cases", {})}
    if agr:
        print(f"agreement of predicted (text, decision) sets with the reference: mean {np.mean(list(agr.values())):.1f}% "
              f"over {len(agr)} cases; lowest:", sorted(agr.items(), key=lambda kv: kv[1])[:5])
else:
    print("No llm_p5/reference/system_check_reference.json: nothing to compare against.")

## DEV run
Resumable: interrupt (stop the cell) and re-run to continue. `MAX_HOURS_DEV` stops starting new docs after the budget. The runtime
projection is an **estimate** (windows x median s/window from the system check; real windows are longer).

In [ ]:
ROWS = {}


def run_split(sp, max_hours):
    docs, med = SELECTED[sp], STATS["median_latency_s"]
    for src in sorted({d["source"] for d in docs}):
        pr = project_runtime([d for d in docs if d["source"] == src], med, WINDOW, OVERLAP)
        print(f"{sp}/{src}: {pr['n_docs']} docs, {pr['n_windows']} windows, estimated ~{pr['est_seconds'] / 60:.0f} min (estimate)")
    pr = project_runtime(docs, med, WINDOW, OVERLAP)
    print(f"{sp} total: {pr['n_docs']} docs, {pr['n_windows']} windows, estimated ~{pr['est_seconds'] / 3600:.1f} h (estimate)"
          + (f"; budget {max_hours} h" if max_hours else ""))
    rows = run_detection(docs, RESULTS_DIR / f"qwen_standalone_{sp}.jsonl", chat_fn, META, "standalone", None,
                         WINDOW, OVERLAP, max_seconds=None if max_hours is None else max_hours * 3600)
    print(f"{sp}: {len(rows)} of {len(docs)} sampled docs completed; windows={sum(r['n_windows'] for r in rows)} "
          f"truncated={sum(r['n_truncated'] for r in rows)} invalid={sum(r['n_invalid'] for r in rows)} "
          f"unmatched strings={sum(r['n_unmatched'] for r in rows)}")
    return rows


if "dev" in RUN_SPLITS:
    ROWS["dev"] = run_split("dev", MAX_HOURS_DEV)
else:
    print("dev not in RUN_SPLITS: skipped")

In [ ]:
if ROWS.get("dev"):
    print_recovery_report(ROWS["dev"])

## Dev metrics
Whitespace excluded. *Span recall* = gold redact spans fully covered by REDACT predictions; *span precision* = REDACT predictions fully
inside gold redact spans; *F2* = 5PR/(4P+R); *char over-redaction %* = non-PII chars flagged / all chars flagged;
*non_pii_flag_rate* = non-PII chars flagged / all non-PII chars; *char leakage %* = uncovered gold chars / all gold chars;
*doc leakage* = docs with an incompletely covered gold span / docs with gold PII; *doc false alarm* = docs without gold PII
that got a flag / docs without gold PII; *keep_false_redaction_rate* = gold keep spans overlapped by a REDACT prediction / gold keep spans.
Sources are never pooled; 95% bootstrap CIs over documents (1000, seed 42); `n` in every row.

In [ ]:
TABLES, LABEL_TABLES = {}, {}      # [split][scope]


def split_tables(sp):
    rows = ROWS.get(sp)
    if not rows:
        print(f"no {sp} rows: run the {sp.upper()} run first")
        return
    TABLES[sp], LABEL_TABLES[sp] = {}, {}
    for scope in ["direct+quasi", "direct"]:
        s, l = score_table(rows, DOCS_BY_ID, scope, TAB_ANNOTATOR, sources=["tab"] if TABLES[sp] else None,
                           n_boot=N_BOOT, seed=SEED)
        TABLES[sp][scope], LABEL_TABLES[sp][scope] = s, l
        print(f"=== {sp}, scope={scope} (applies to TAB; CRAPII is scope-free), annotator={TAB_ANNOTATOR} ===")
        print(fmt_ci(s).to_string(index=False))
    for scope, l in LABEL_TABLES[sp].items():
        print(f"=== {sp}: recall by gold label / keep_false_redaction_rate by gold label, scope={scope} ===")
        out = l.assign(value=lambda x: x.value.round(3), lo=lambda x: x.lo.round(3), hi=lambda x: x.hi.round(3))
        print(out[["source", "split", "kind", "label", "n", "value", "lo", "hi"]].to_string(index=False))


def plot_split(sp):
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
    for ax, scope in zip(axes, TABLES[sp]):
        t = TABLES[sp][scope]
        for off, k in zip((-0.2, 0.2), ("span_recall", "span_precision")):
            ax.bar([i + off for i in range(len(t))], t[k], 0.4, label=k,
                   yerr=[t[k] - t[k + "_lo"], t[k + "_hi"] - t[k]], capsize=3)
        ax.set_xticks(range(len(t)), [f"{a}/{b}" for a, b in zip(t.source, t.split)])
        ax.set_title(f"{sp}, TAB scope: {scope}")
    axes[0].legend(); axes[0].set_ylim(0, 1.05)
    plt.tight_layout(); plt.show()


split_tables("dev")
if "dev" in TABLES:
    plot_split("dev")

### Dev error examples
Worst-10 per source (context truncated to 120 chars). Misses = gold redact spans not covered; false redactions = gold keep spans hit by a REDACT prediction.

In [ ]:
if ROWS.get("dev"):
    MISSES, FALSE_RED = worst_errors(ROWS["dev"], DOCS_BY_ID, TAB_SCOPE, TAB_ANNOTATOR, k=10)
    pd.set_option("display.max_colwidth", 120)
    print("--- worst misses ---");           print(MISSES.drop(columns=["split"], errors="ignore").to_string(index=False))
    print("--- worst false redactions of keep spans ---"); print(FALSE_RED.drop(columns=["split"], errors="ignore").to_string(index=False))

## Compare machines on dev
Set `OTHER_DEV_FILE` (a `qwen_standalone_dev.jsonl` from another machine, placed in `llm_p5/results/incoming/`) to compare. Skipped when `None`.

> At temperature 0 with a fixed seed, agreement across hardware is likely but **not guaranteed** (different kernels and floating-point
> order can change a token). Disagreement on a few docs is expected; large disagreement or a digest/prompt mismatch is not.

In [ ]:
if OTHER_DEV_FILE is None:
    print("OTHER_DEV_FILE is None: machine comparison skipped.")
else:
    p = Path(OTHER_DEV_FILE)
    if not p.exists():
        p = INCOMING_DIR / p.name
    assert p.exists(), f"{OTHER_DEV_FILE} not found (put the other machine's dev results file in llm_p5/results/incoming/)"
    mine = latest_rows(load_jsonl(RESULTS_DIR / "qwen_standalone_dev.jsonl"))
    other = latest_rows(load_jsonl(p))
    cmp = check_comparable(mine, other)
    for w in cmp["warnings"]:
        print("WARNING:", w)
    common = [i for i in cmp["common"] if i in DOCS_BY_ID]
    if not cmp["valid"]:
        print("COMPARISON INVALID (model digest mismatch, or no shared docs): not computing agreement.")
    else:
        A = [r for r in mine if r["doc_id"] in set(common)]
        B = [r for r in other if r["doc_id"] in set(common)]
        summ, per_doc = run_agreement(A, B)
        print("agreement on", summ["n_common"], "common docs:", {k: round(v, 4) for k, v in summ.items() if k != "n_common"})
        print(per_doc.groupby("source").agg(n=("doc_id", "size"), jaccard_all=("jaccard_all", "mean"),
              identical=("identical", "mean"), jaccard_redact=("jaccard_redact", "mean")).round(4).to_string())
        sa, _ = score_table(A, DOCS_BY_ID, TAB_SCOPE, TAB_ANNOTATOR, n_boot=50, seed=SEED)
        sb, _ = score_table(B, DOCS_BY_ID, TAB_SCOPE, TAB_ANNOTATOR, n_boot=50, seed=SEED)
        print("--- metrics on common docs (A = this machine, B = other) ---")
        print(metrics_diff(sa, sb).round(4).to_string(index=False))
        other_gpu = json.loads((INCOMING_DIR / Path(OTHER_CHECK_FILE).name).read_text(encoding="utf-8")).get("gpu_share") if OTHER_CHECK_FILE else None
        spd = pd.DataFrame({"A (this machine)": speed_stats(A, GPU), "B (other)": speed_stats(B, other_gpu)})
        print("--- speed ---"); print(spd.to_string())
        import matplotlib.pyplot as plt
        la, lb = f"A: {spd.loc['chip', 'A (this machine)']}", f"B: {spd.loc['chip', 'B (other)']}"
        fig, ax = plt.subplots(1, 3, figsize=(12, 3.4))
        for a, key, ttl in zip(ax[:2], ["median_s_per_window", "median_tokens_per_s"], ["median s / window", "tokens / s"]):
            a.bar([0, 1], [spd.loc[key, "A (this machine)"], spd.loc[key, "B (other)"]], color=["#4477aa", "#ee6677"])
            a.set_xticks([0, 1], [la[:22], lb[:22]], fontsize=7); a.set_title(ttl)
        ax[2].bar([0, 1, 2], [summ["share_identical"], summ["mean_jaccard_all"], summ["mean_jaccard_redact"]], color="#228833")
        ax[2].set_xticks([0, 1, 2], ["identical docs", "Jaccard all", "Jaccard REDACT"], fontsize=7)
        ax[2].set_ylim(0, 1.05); ax[2].set_title(f"agreement (n={summ['n_common']} docs)")
        plt.tight_layout(); out_png = RESULTS_DIR / "machine_comparison_dev.png"
        plt.savefig(out_png, dpi=130); plt.show()
        print("saved", out_png.relative_to(ROOT), "(no document text on it)")

## TRAIN run
Same resumable runner on the train sample. `MAX_HOURS_TRAIN` (hours) stops starting new docs once the budget is used; the finished docs are a
stratified random subset (see the prefix note under Setup). Stop the cell any time and re-run to resume.

In [ ]:
if "train" in RUN_SPLITS:
    ROWS["train"] = run_split("train", MAX_HOURS_TRAIN)
else:
    print("train not in RUN_SPLITS: skipped")

In [ ]:
if ROWS.get("train"):
    print_recovery_report(ROWS["train"])

## Train benchmark (pre-tuning)
> Train is for **error mining and choosing few-shot examples**. Any doc used as a few-shot example must be excluded from train scoring
> later. These numbers are the **pre-tuning reference**, not the report numbers.

In [ ]:
split_tables("train")
if "train" in TABLES:
    plot_split("train")

In [ ]:
if ROWS.get("train"):
    EC = error_counts(ROWS["train"], DOCS_BY_ID, TAB_SCOPE, TAB_ANNOTATOR)
    EC.to_csv(RESULTS_DIR / "train_error_mining.csv", index=False, encoding="utf-8", lineterminator="\n")
    print(f"=== error counts by gold label (TAB scope {TAB_SCOPE}) ===");  print(EC.to_string(index=False))
    W_MISS, W_FALSE = worst_errors(ROWS["train"], DOCS_BY_ID, TAB_SCOPE, TAB_ANNOTATOR, k=30)
    worst = pd.concat([W_MISS.assign(kind="miss"), W_FALSE.assign(kind="false_redaction")], ignore_index=True)
    worst.to_csv(RESULTS_DIR / "train_worst30_errors.csv", index=False, encoding="utf-8", lineterminator="\n")
    print(f"worst-30 per source: {len(W_MISS)} misses, {len(W_FALSE)} false redactions written to "
          "llm_p5/results/train_worst30_errors.csv (contains document text: keep it out of git)")
else:
    print("no train rows yet")

## Optional: baseline scoring (dev and train)
JSONL per split, one doc per line: `{doc_id, spans:[{start,end,text,label,decision?}]}` (e.g. GLiNER output; missing `decision` = REDACT).
Scored on the same docs as the Qwen run with the same metrics.

In [ ]:
BASELINE_PATHS = {"dev": None, "train": None}      # e.g. {"dev": "results/preds/gliner_dev.jsonl", "train": ...}
BASELINE_NAME = "baseline"
SIDE = {}
for sp, bp in BASELINE_PATHS.items():
    if not bp or not ROWS.get(sp):
        continue
    base_rows = [r for r in load_baseline(bp) if r["doc_id"] in DOCS_BY_ID]
    have = {r["doc_id"] for r in base_rows}
    print(f"{sp}: baseline covers {len(have)} of {len(ROWS[sp])} Qwen-run docs; scoring only the shared docs")
    shared = [r for r in ROWS[sp] if r["doc_id"] in have]
    q, _ = score_table(shared, DOCS_BY_ID, TAB_SCOPE, TAB_ANNOTATOR, n_boot=N_BOOT, seed=SEED)
    b, _ = score_table(base_rows, DOCS_BY_ID, TAB_SCOPE, TAB_ANNOTATOR, n_boot=N_BOOT, seed=SEED)
    SIDE[sp] = pd.concat([fmt_ci(q).assign(system=MODEL_TAG), fmt_ci(b).assign(system=BASELINE_NAME)]).sort_values(["source", "split", "system"])
    print(SIDE[sp].to_string(index=False))
if not SIDE:
    print("No BASELINE_PATHS set (or no rows yet): skipped.")

## Optional: review mode (disabled by default; architecture undecided)
A candidates JSONL `{doc_id, spans:[{start,end,text,label,score}]}` is shown per window as numbered candidates; the LLM returns
REDACT/KEEP per id and may list missed strings. No gold information is in the prompt. Rows carry `mode: "review"`.

In [ ]:
REVIEW_ENABLED = False
CANDIDATES_PATHS = {"dev": None, "train": None}
if REVIEW_ENABLED:
    rmeta = {**META, "prompt_hash": prompt_hash("review")}
    for sp, cp in CANDIDATES_PATHS.items():
        if not cp or sp not in SELECTED:
            continue
        cand = {r["doc_id"]: r["spans"] for r in load_jsonl(cp)}
        rr = run_detection(SELECTED[sp], RESULTS_DIR / f"qwen_review_{sp}.jsonl", chat_fn, rmeta, "review", cand, WINDOW, OVERLAP)
        s, _ = score_table(rr, DOCS_BY_ID, TAB_SCOPE, TAB_ANNOTATOR, n_boot=N_BOOT, seed=SEED)
        print(sp); print(fmt_ci(s).to_string(index=False))
else:
    print("Review mode disabled.")

## Export
Aggregates only (no document text): `summary_<chip>_<ram>gb.md` and `.csv` for dev and train.

In [ ]:
tag = MACHINE_TAG
frames = []
for sp, scopes in TABLES.items():
    frames.append(scopes["direct+quasi"].assign(scope_key="direct+quasi"))
    frames.append(scopes["direct"][scopes["direct"].source == "tab"].assign(scope_key="direct"))
assert frames, "nothing to export yet: run the DEV (and TRAIN) run and metrics first"
CSV = pd.concat(frames).assign(model=MODEL_TAG, prompt_version=PROMPT_VERSION, prompt_hash=PHASH,
                               prompt_modified=PROMPT_MODIFIED, annotator=TAB_ANNOTATOR)
csv_path = RESULTS_DIR / f"summary_{tag}.csv"
CSV.to_csv(csv_path, index=False, encoding="utf-8", lineterminator="\n")

lines = [f"# P5 Qwen summary ({tag})", "",
         f"- model `{MODEL_TAG}` digest `{INFO['digest']}`; prompt {PROMPT_VERSION} hash `{PHASH}` (modified: {PROMPT_MODIFIED}); Ollama {OLLAMA_VER}",
         f"- hardware: {HW['chip']}, {HW['ram_gb']} GB RAM, GPU {HW['gpu']['name']} ({HW['gpu']['vram_gb']} GB, driver {HW['gpu']['driver']}), {HW['os']}, Python {HW['python']}",
         f"- splits: {list(ROWS)}; sample fingerprints {SAMPLE_FP}; TAB annotator {TAB_ANNOTATOR}; windows={WINDOW}/{OVERLAP}; seed={SEED}",
         f"- system check: median {STATS['median_latency_s']:.2f}s/window, {STATS['median_tokens_per_s']:.1f} tok/s, "
         f"generation {STATS['median_gen_s']:.2f}s, prompt eval {STATS['median_prompt_eval_s']:.2f}s, overhead {STATS['median_overhead_s']:.2f}s, "
         f"valid JSON {STATS['valid_json_rate']:.2%}, decision agreement {STATS['decision_agreement']:.2%}, GPU share {GPU}"]
for sp, rows in ROWS.items():
    lines.append(f"- {sp}: {len(rows)} of {len(SELECTED[sp])} sampled docs completed, windows={sum(r['n_windows'] for r in rows)} "
                 f"truncated={sum(r['n_truncated'] for r in rows)} invalid={sum(r['n_invalid'] for r in rows)} "
                 f"unmatched strings={sum(r['n_unmatched'] for r in rows)}, wall time {sum(r['latency_s'] for r in rows) / 3600:.2f} h")
lines.append("")
for sp, scopes in TABLES.items():
    for scope in ["direct+quasi", "direct"]:
        lines += [f"## {sp}: metrics (TAB scope {scope}; CRAPII is scope-free)", "", "```", fmt_ci(scopes[scope]).to_string(index=False), "```", ""]
md_path = RESULTS_DIR / f"summary_{tag}.md"
md_path.write_text("\n".join(lines), encoding="utf-8", newline="\n")
print("wrote:", md_path.relative_to(ROOT), csv_path.relative_to(ROOT), CHECK_PATH.relative_to(ROOT), sep="\n  ")